## 434_samtrafiken-vechicle-id
* [#434](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/434)
* denna Notebook [434_samtrafiken-vechicle-id.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/434_samtrafiken-vechicle-id.ipynb)



In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-07-01 00:27


In [2]:
import os
import requests

from google.transit import gtfs_realtime_pb2

API_KEY = os.environ["GTFSSweden3"]

URL = (
    "https://opendata.samtrafiken.se/gtfs-rt/sweden/VehiclePositions.pb"
    f"?key={API_KEY}"
)

r = requests.get(URL)
r.raise_for_status()

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(r.content)

count = 0

for entity in feed.entity:

    if not entity.HasField("vehicle"):
        continue

    v = entity.vehicle

    print("=" * 60)

    print("vehicle.id      :", v.vehicle.id)

    if v.vehicle.label:
        print("vehicle.label   :", v.vehicle.label)

    if v.vehicle.license_plate:
        print("license_plate   :", v.vehicle.license_plate)

    print("trip_id         :", v.trip.trip_id)
    print("route_id        :", v.trip.route_id)

    print("lat/lon         :", v.position.latitude, v.position.longitude)

    count += 1
    if count == 20:
        break

HTTPError: 403 Client Error:  for url: https://opendata.samtrafiken.se/gtfs-rt/sweden/VehiclePositions.pb?key=f2277323909847378fb5951b57fbce03

In [3]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    agencies = pd.read_csv(z.open("agency.txt"))

print(agencies)


              agency_id               agency_name                agency_url  \
0                    50  Samtrafiken i Sverige AB  https://www.resrobot.se/   
1    500000000000000043          Trafikverket RDB  https://www.resrobot.se/   
2    500000000000000107           Vy flygbussarna  https://www.resrobot.se/   
3    500000000000000109                       109  https://www.resrobot.se/   
4    500000000000000110                       110  https://www.resrobot.se/   
..                  ...                       ...                       ...   
244  505000000000000764          Tåg i Bergslagen  https://www.resrobot.se/   
245  505000000000000765              Krösatåg KLT  https://www.resrobot.se/   
246  505000000000000766                   Norrtåg  https://www.resrobot.se/   
247  505000000000000767                 Vy Tåg AB  https://www.resrobot.se/   
248  505000000000000768               LT X-trafik  https://www.resrobot.se/   

      agency_timezone agency_lang  agency_fare_url 

In [4]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    agencies = pd.read_csv(z.open("agency.txt"))
    routes = pd.read_csv(z.open("routes.txt"))

# Leta efter Waxholmsbolaget
waxholm = agencies[
    agencies["agency_name"].str.contains("wax", case=False, na=False)
]

print(waxholm)

wax_routes = routes.merge(agencies, on="agency_id")

wax_routes = wax_routes[
    wax_routes["agency_name"].str.contains("wax", case=False, na=False)
]

print(wax_routes[[
    "route_id",
    "route_short_name",
    "route_long_name",
    "route_type",
    "agency_name"
]])

              agency_id                    agency_name  \
159  505000000000000606  Waxholmsbolaget Ångfartygs AB   

                   agency_url   agency_timezone agency_lang  agency_fare_url  
159  https://www.resrobot.se/  Europe/Stockholm          sv              NaN  
Empty DataFrame
Columns: [route_id, route_short_name, route_long_name, route_type, agency_name]
Index: []


In [5]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    routes = pd.read_csv(z.open("routes.txt"))

print(routes.columns.tolist())

print("\nAntal routes:", len(routes))

print("\nAgency IDs som används:")

print(routes["agency_id"].value_counts())

['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_desc']

Antal routes: 7575

Agency IDs som används:
agency_id
505000000000000014    1672
505000000000000100     576
500000000000000790     567
505000000000000001     534
505000000000000174     383
                      ... 
505000000000000120       1
500000000000000154       1
505000000000000145       1
505000000000000140       1
505000000000000111       1
Name: count, Length: 81, dtype: int64


In [6]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    agencies = pd.read_csv(z.open("agency.txt"))
    routes = pd.read_csv(z.open("routes.txt"))

routes = routes.merge(agencies, on="agency_id", how="left")

# Visa alla färjelinjer
ferries = routes[routes["route_type"] == 4]

print(f"Antal färjelinjer: {len(ferries)}")

print(
    ferries[
        [
            "agency_id",
            "agency_name",
            "route_id",
            "route_short_name",
            "route_long_name",
        ]
    ].head(50)
)

Antal färjelinjer: 0
Empty DataFrame
Columns: [agency_id, agency_name, route_id, route_short_name, route_long_name]
Index: []


In [7]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    routes = pd.read_csv(z.open("routes.txt"))

print(routes["route_type"].value_counts().sort_index())


route_type
100       66
101      459
103      123
105       20
106     2871
401        7
700     3363
714       12
900       30
1000     103
1501     521
Name: count, dtype: int64


In [8]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    routes = pd.read_csv(z.open("routes.txt"))

for rt in sorted(routes.route_type.unique()):
    print("\n========== route_type", rt, "==========")
    cols = ["route_short_name", "route_long_name"]

    if "route_desc" in routes.columns:
        cols.append("route_desc")

    print(routes[routes.route_type == rt][cols].head(20).to_string(index=False))
    


========== route_type 100 ==========
route_short_name                                   route_long_name  route_desc
            1079                                          Krösatåg         NaN
            1085                                          Krösatåg         NaN
            1086                                          Krösatåg         NaN
            1087                                          Krösatåg         NaN
            1088                                          Krösatåg         NaN
            1089                                          Krösatåg         NaN
              70                                               NaN         NaN
              71                                               NaN         NaN
              74                                               NaN         NaN
           Anrop                                               270         NaN
               1                                                SJ         NaN
             3

In [9]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    routes = pd.read_csv(z.open("routes.txt"))

ferries = routes[routes.route_type == 1000]

print(ferries[[
    "route_id",
    "agency_id",
    "route_short_name",
    "route_long_name"
]].to_string(index=False))


            route_id          agency_id route_short_name                   route_long_name
    9011001008000000 505000000000000001               80                               NaN
9011001008000000-80X 505000000000000001              80X                               NaN
    9011001008200000 505000000000000001               82                   Djurgårdsfärjan
    9011001008400000 505000000000000001               84                               NaN
    9011001008900000 505000000000000001               89                               NaN
    9011003090000000 505000000000000003              900                       Gräsöfärjan
    9011005077500000 505000000000000005              775                   Skärgårdstrafik
    9011008098500000 505000000000000008            Dessi                       Färja Dessi
    9011010090800000 505000000000000010              908                               NaN
    9011010091000000 505000000000000010              910                               NaN

In [10]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    agencies = pd.read_csv(z.open("agency.txt"))

print(
    agencies[
        agencies.agency_id == 500000000000000114
    ]
)

            agency_id agency_name                agency_url   agency_timezone  \
5  500000000000000114         114  https://www.resrobot.se/  Europe/Stockholm   

  agency_lang  agency_fare_url  
5          sv              NaN  


In [11]:
from zipfile import ZipFile
import pandas as pd
from google.transit import gtfs_realtime_pb2

# ----- Läs GTFS static -----

with ZipFile("sweden_api.zip") as z:
    trips = pd.read_csv(
        z.open("trips.txt"),
        dtype={"trip_id": str, "route_id": str},
        low_memory=False,
    )

    routes = pd.read_csv(
        z.open("routes.txt"),
        dtype={"route_id": str},
    )

trips = trips[["trip_id", "route_id"]]
routes = routes[["route_id", "route_short_name", "route_long_name", "route_type"]]

# ----- Läs VehiclePositions -----

feed = gtfs_realtime_pb2.FeedMessage()

with open("VehiclePositions.pb", "rb") as f:
    feed.ParseFromString(f.read())

rows = []

for entity in feed.entity:

    if not entity.HasField("vehicle"):
        continue

    v = entity.vehicle
    rows.append({
        "vehicle_id": str(v.vehicle.id),
        "trip_id": str(v.trip.trip_id),
    })

vehicles = pd.DataFrame(rows)

# ----- Koppla ihop -----

df = (
    vehicles
      .merge(trips, on="trip_id", how="left")
      .merge(routes, on="route_id", how="left")
)

print(df.sort_values("route_short_name").head(200))

df.to_csv("vehicle_routes.csv", index=False)

            vehicle_id            trip_id          route_id route_short_name  \
683   9031001001004023  14010000686337262  9011001000100000                1   
893   9031001001004106  14010000686338040  9011001000100000                1   
245   9031001001004020  14010000686336803  9011001000100000                1   
1044  9031001001004003  14010000717717327  9011001000100000                1   
1278  9031001001004019  14010000724503990  9011001000100000                1   
...                ...                ...               ...              ...   
18    9031001003002738  14010000689190197  9011001016700000              167   
1012  9031001003005589  14010000689193126  9011001016700000              167   
875   9031001003005531  14010000689191094  9011001016700000              167   
159   9031001003002745  14010000689193924  9011001016700000              167   
65    9031001002500043  14010100722680590  9011001001700000               17   

     route_long_name  route_type  
683 

In [12]:
ferries = df[df["route_type"] == 1000].copy()

ferries = ferries.sort_values(["route_short_name", "vehicle_id"])

print(ferries[
    ["vehicle_id",
     "route_short_name",
     "route_id"]
])

ferries.to_csv("ferries.csv", index=False)

            vehicle_id route_short_name              route_id
1035  9031001000500504               11      9011114001100000
246   9031001000500538               12      9011114001200000
713   9031001000500539               12      9011114001200000
1220  9031001000500520               13      9011114001300000
216   9031001000500547               13      9011114001300000
1335  9031001006500776               13      9011114001300000
1292  9031001000500536               14      9011114001400000
1363  9031001000500540               14      9011114001400000
390   9031001000500546               14      9011114001400000
436   9031001000500681               14      9011114001400000
379   9031001000500736               14      9011114001400000
274   9031001000500513               16      9011114001600000
923   9031001000500549               16      9011114001600000
986   9031001000500519               17      9011114001700000
188   9031001008500737               17      9011114001700000
479   90

In [13]:
route3 = ferries[ferries["route_short_name"] == "3"]
route3

,vehicle_id,trip_id,route_id,route_short_name,route_long_name,route_type
96,9031001000500555,14010000726426404,9011114000300000,3,NaN,1000.0


In [15]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    stop_times = pd.read_csv(z.open("stop_times.txt"))

stop_times.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,pickup_booking_rule_id,drop_off_booking_rule_id
0,400000000000009622,04:08:00,04:08:00,9022050001617006,1,Uppsala central,0,1,NaN,1,NaN,NaN
1,400000000000009622,04:10:00,04:10:00,9022050001618003,2,Uppsala central,0,0,NaN,1,NaN,NaN
2,400000000000009622,04:15:00,04:15:00,9022050000759010,3,Uppsala central,0,0,NaN,1,NaN,NaN
3,400000000000009622,04:18:00,04:18:00,9022050000781003,4,Uppsala central,0,0,NaN,1,NaN,NaN
4,400000000000009622,04:21:00,04:21:00,9022050011423003,5,Uppsala central,0,0,NaN,1,NaN,NaN


In [16]:
trip_id = "14010000726426404"

trip = (
    stop_times[stop_times.trip_id == trip_id]
    .sort_values("stop_sequence")
)

trip

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,pickup_booking_rule_id,drop_off_booking_rule_id


In [17]:
with ZipFile("sweden_api.zip") as z:
    stops = pd.read_csv(z.open("stops.txt"))

stops.head()

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station,platform_code
0,1,Stockholm Centralstation,59.331537,18.054943,1,NaN,NaN
1,2,Göteborg Centralstation,57.709299,11.973659,1,NaN,NaN
2,3,Malmö Centralstation,55.608777,13.000216,1,NaN,NaN
3,4,Alvesta station,56.898911,14.556790,1,NaN,NaN
4,5,Uppsala Centralstation,59.857739,17.647666,1,NaN,NaN


In [18]:
trip_stops = (
    stop_times
      .merge(stops, on="stop_id")
      .sort_values(["trip_id","stop_sequence"])
)

trip_stops.head(20)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,pickup_booking_rule_id,drop_off_booking_rule_id,stop_name,stop_lat,stop_lon,location_type,parent_station,platform_code
1548097,14010000509102393,23:47:00,23:47:00,9022050010353004,1,Södertälje centrum,3,1,0.00,1,NaN,NaN,Hovsjöskolan,59.175203,17.603916,0,10353.0,4
1548098,14010000509102393,23:47:59,23:47:59,9022050011476002,2,Södertälje centrum,3,3,295.62,0,NaN,NaN,Hovsjö centrum,59.176716,17.608072,0,11476.0,2
1548099,14010000509102393,23:49:36,23:49:36,9022050008498002,3,Södertälje centrum,3,3,959.14,0,NaN,NaN,Vibergen,59.178788,17.614674,0,8498.0,2
1548100,14010000509102393,23:50:55,23:50:55,9022050011475002,4,Södertälje centrum,3,3,1526.70,0,NaN,NaN,Saltskogs centrum,59.178910,17.620653,0,11475.0,2
1548101,14010000509102393,23:51:40,23:51:40,9022050008497002,5,Södertälje centrum,3,3,1821.35,0,NaN,NaN,Förmansvägen,59.181006,17.623194,0,8497.0,2
1548102,14010000509102393,23:53:23,23:53:23,9022050010354002,6,Södertälje centrum,3,3,2533.20,0,NaN,NaN,Scania centralkontor,59.183046,17.631200,0,10354.0,2
1548103,14010000509102393,23:55:19,23:55:19,9022050008484002,7,Södertälje centrum,3,3,3489.52,0,NaN,NaN,Värdsholmsgatan,59.187481,17.631763,0,8484.0,2
1548104,14010000509102393,23:58:00,23:58:00,9022050012578008,8,Södertälje centrum,1,3,4306.45,1,NaN,NaN,Södertälje centrum,59.192071,17.627843,0,12578.0,F
1548105,14010000509110182,24:17:00,24:17:00,9022050010353004,1,Södertälje centrum,3,1,0.00,1,NaN,NaN,Hovsjöskolan,59.175203,17.603916,0,10353.0,4
1548106,14010000509110182,24:17:59,24:17:59,9022050011476002,2,Södertälje centrum,3,3,295.62,0,NaN,NaN,Hovsjö centrum,59.176716,17.608072,0,11476.0,2


In [19]:
with ZipFile("sweden_api.zip") as z:
    trips = pd.read_csv(z.open("trips.txt"))
    routes = pd.read_csv(z.open("routes.txt"))

route3 = routes[routes.route_short_name=="3"]

route3

/var/folders/fd/md6r13sj0wsbg_6_xl160d300000gn/T/ipykernel_52670/4080688729.py:2: DtypeWarning: Columns (0,4) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(z.open("trips.txt"))


,route_id,agency_id,route_short_name,route_long_name,route_type,route_desc
6,9011001000300000,505000000000000001,3,NaN,700,NaN
536,9011003000300000,505000000000000003,3,NaN,700,NaN
649,9011004000300000,505000000000000004,3,NaN,700,NaN
672,9011004014300000,505000000000000004,3,NaN,700,NaN
673,9011004016300000,505000000000000004,3,NaN,700,NaN
680,9011004018300000,505000000000000004,3,NaN,700,NaN
887,9011005000300000,505000000000000005,3,NaN,900,NaN
912,9011005020300000,505000000000000005,3,NaN,700,NaN
997,9011006000300000,505000000000000006,3,NaN,700,NaN
1087,9011007000300000,505000000000000007,3,NaN,700,NaN


In [20]:
print(stop_times.columns.tolist())
print(trips.columns.tolist())

['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled', 'timepoint', 'pickup_booking_rule_id', 'drop_off_booking_rule_id']
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'shape_id', 'samtrafiken_internal_trip_number']


In [21]:
route3 = trips[trips.route_id == "9011114000300000"]

route3[
    [
        "trip_id",
        "trip_short_name",
        "samtrafiken_internal_trip_number",
        "service_id",
        "direction_id"
    ]
].sort_values(
    "samtrafiken_internal_trip_number"
)

,trip_id,trip_short_name,samtrafiken_internal_trip_number,service_id,direction_id
406594,14010000726445298,NaN,300,811,1
406606,14010000729656166,NaN,301,1647,1
406608,14010000729656390,NaN,301,645,1
406612,14010000729656226,NaN,301,3954,1
406588,14010000726426339,NaN,301,414,1
...,...,...,...,...,...
406592,14010000726426187,NaN,379,637,1
406624,14010000721785463,NaN,380,637,0
406596,14010000729172203,NaN,381,637,1
406619,14010000721785443,NaN,382,637,0


In [23]:
route3.to_csv("route3.csv")

In [24]:
route3 = trips[trips["route_id"] == "9011114000300000"]

print("Trips:", len(route3))

route3[
    [
        "trip_id",
        "service_id",
        "direction_id",
        "trip_short_name",
        "samtrafiken_internal_trip_number"
    ]
].sort_values(
    ["direction_id", "samtrafiken_internal_trip_number"]
)

Trips: 72


,trip_id,service_id,direction_id,trip_short_name,samtrafiken_internal_trip_number
406629,14010000725132390,414,0,NaN,302
406646,14010000729656520,677,0,NaN,302
406639,14010000725132327,379,0,NaN,304
406635,14010000725132367,414,0,NaN,306
406645,14010000729656545,677,0,NaN,306
...,...,...,...,...,...
406576,14010000725131569,637,1,NaN,373
406607,14010000729656203,637,1,NaN,375
406598,14010000729656180,637,1,NaN,377
406592,14010000726426187,637,1,NaN,379


In [25]:
trip305 = route3[
    route3["samtrafiken_internal_trip_number"] == 305
]

trip305

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,shape_id,samtrafiken_internal_trip_number
406616,9011114000300000,810,14010100726445298,NaN,NaN,1,6014010000721783677,305


In [26]:
trip_id = trip305.iloc[0]["trip_id"]

(
    stop_times[stop_times.trip_id == trip_id]
      .merge(
          stops[["stop_id", "stop_name"]],
          on="stop_id"
      )
      .sort_values("stop_sequence")
)[[
    "stop_sequence",
    "stop_name",
    "arrival_time",
    "pickup_booking_rule_id",
    "drop_off_booking_rule_id"
]]

,stop_sequence,stop_name,arrival_time,pickup_booking_rule_id,drop_off_booking_rule_id
0,1,Grenadjärbryggan,06:45:00,NaN,NaN
1,2,Ramsöberg,06:50:00,NaN,NaN
2,3,Knutsholmen,07:00:00,NaN,NaN
3,4,Skogsö södra,07:05:00,NaN,NaN
4,5,Hästholmen,07:07:00,NaN,NaN
5,6,Skogsö udde,07:10:00,NaN,NaN
6,7,Norra Lagnö,07:20:00,NaN,NaN


In [27]:
trip305 = (
    stop_times
      .query("trip_id == @trip_id")
      .merge(stops,on="stop_id")
      .sort_values("stop_sequence")
)

trip305

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,pickup_booking_rule_id,drop_off_booking_rule_id,stop_name,stop_lat,stop_lon,location_type,parent_station,platform_code
0,14010100726445298,06:45:00,06:45:00,9022050013101001,1,Gåshaga via Norra Lagnö,3,1,0.00,1,NaN,NaN,Grenadjärbryggan,59.390903,18.433926,0,13101.0,1
1,14010100726445298,06:50:00,06:50:00,9022050013093001,2,Gåshaga via Norra Lagnö,3,3,1301.19,1,NaN,NaN,Ramsöberg,59.384360,18.418887,0,13093.0,1
2,14010100726445298,07:00:00,07:00:00,9022050013094001,3,Gåshaga via Norra Lagnö,3,3,2786.40,1,NaN,NaN,Knutsholmen,59.376485,18.435003,0,13094.0,1
3,14010100726445298,07:05:00,07:05:00,9022050013096001,4,Gåshaga via Norra Lagnö,3,3,4300.70,1,NaN,NaN,Skogsö södra,59.368334,18.442454,0,13096.0,1
4,14010100726445298,07:07:00,07:07:00,9022050013095001,5,Gåshaga via Norra Lagnö,3,3,4772.60,1,NaN,NaN,Hästholmen,59.365172,18.441042,0,13095.0,1
5,14010100726445298,07:10:00,07:10:00,9022050013097001,6,Gåshaga via Norra Lagnö,3,3,5260.95,1,NaN,NaN,Skogsö udde,59.366367,18.433891,0,13097.0,1
6,14010100726445298,07:20:00,07:20:00,9022050013001001,7,Gåshaga via Norra Lagnö,1,3,7544.44,1,NaN,NaN,Norra Lagnö,59.355297,18.411543,0,13001.0,1


In [28]:
trip305["pdf_flag"] = None

In [37]:
with ZipFile("sweden_api.zip") as z: 
     files = sorted(z.namelist())
for f in files:
    if "booking" in f.lower():
        print(f)

booking_rules.txt


In [38]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    booking_rules = pd.read_csv(z.open("booking_rules.txt"))

booking_rules.head(100)

,booking_rule_id,booking_type,prior_notice_duration_min,prior_notice_last_day,prior_notice_last_time,message,phone_number
0,14010000696807824,1,120.0,NaN,NaN,Resan måste beställas på sl.se senast 2 timmar...,NaN
1,14010000696809111,1,120.0,NaN,NaN,Resan måste beställas på sl.se senast 2 timmar...,NaN
2,14010000696811043,1,120.0,NaN,NaN,Resan måste beställas på sl.se senast 2 timmar...,NaN
3,14010000697510764,1,120.0,NaN,NaN,Resan måste beställas på sl.se senast 2 timmar...,NaN
4,14010000697638602,1,120.0,NaN,NaN,Resan måste beställas på sl.se senast 2 timmar...,NaN
...,...,...,...,...,...,...,...
95,40000000000000013,2,NaN,1.0,20:00:00,Måste förbeställas senast kl. 20:00 dagen före...,0771-22 40 00
96,40000000000000014,2,NaN,0.0,20:00:00,Måste förbeställas senast kl. 20:00 samma dag ...,0771-22 40 00
97,40000000000000015,2,NaN,1.0,20:00:00,Måste förbeställas senast kl. 20:00 dagen före...,0771-22 40 00
98,40000000000000016,2,NaN,0.0,20:00:00,Måste förbeställas senast kl. 20:00 samma dag ...,0771-22 40 00


In [39]:
trip = (
    stop_times[stop_times.trip_id == trip_id]
      .merge(
          stops[["stop_id", "stop_name"]],
          on="stop_id"
      )
      .merge(
          booking_rules,
          left_on="pickup_booking_rule_id",
          right_on="booking_rule_id",
          how="left",
          suffixes=("","_pickup")
      )
      .sort_values("stop_sequence")
)

trip[
    [
        "stop_sequence",
        "stop_name",
        "arrival_time",
        "pickup_type",
        "pickup_booking_rule_id",
        "prior_notice_duration_min",
        "message"
    ]
]

,stop_sequence,stop_name,arrival_time,pickup_type,pickup_booking_rule_id,prior_notice_duration_min,message
0,1,Grenadjärbryggan,06:45:00,3,NaN,NaN,NaN
1,2,Ramsöberg,06:50:00,3,NaN,NaN,NaN
2,3,Knutsholmen,07:00:00,3,NaN,NaN,NaN
3,4,Skogsö södra,07:05:00,3,NaN,NaN,NaN
4,5,Hästholmen,07:07:00,3,NaN,NaN,NaN
5,6,Skogsö udde,07:10:00,3,NaN,NaN,NaN
6,7,Norra Lagnö,07:20:00,1,NaN,NaN,NaN


In [40]:
trip = (
    stop_times[stop_times.trip_id == trip_id]
      .merge(
          stops[["stop_id", "stop_name"]],
          on="stop_id"
      )
      .merge(
          booking_rules,
          left_on="pickup_booking_rule_id",
          right_on="booking_rule_id",
          how="left",
          suffixes=("","_pickup")
      )
      .sort_values("stop_sequence")
)

trip[
    [
        "stop_sequence",
        "stop_name",
        "arrival_time",
        "pickup_type",
        "pickup_booking_rule_id",
        "prior_notice_duration_min",
        "message"
    ]
]

,stop_sequence,stop_name,arrival_time,pickup_type,pickup_booking_rule_id,prior_notice_duration_min,message
0,1,Grenadjärbryggan,06:45:00,3,NaN,NaN,NaN
1,2,Ramsöberg,06:50:00,3,NaN,NaN,NaN
2,3,Knutsholmen,07:00:00,3,NaN,NaN,NaN
3,4,Skogsö södra,07:05:00,3,NaN,NaN,NaN
4,5,Hästholmen,07:07:00,3,NaN,NaN,NaN
5,6,Skogsö udde,07:10:00,3,NaN,NaN,NaN
6,7,Norra Lagnö,07:20:00,1,NaN,NaN,NaN


In [41]:
print(len(booking_rules))

5485


In [42]:
booking_rules[
    [
        "prior_notice_duration_min",
        "prior_notice_last_day",
        "prior_notice_last_time"
    ]
].drop_duplicates()

,prior_notice_duration_min,prior_notice_last_day,prior_notice_last_time
0,120.0,NaN,NaN
35,0.0,NaN,NaN
39,NaN,NaN,NaN
83,NaN,1.0,20:00:00
85,NaN,0.0,20:00:00
217,NaN,1.0,18:00:00
445,NaN,1.0,04:00:00
803,60.0,NaN,NaN
1001,NaN,1.0,22:00:00
1401,180.0,NaN,NaN
